In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("WineQT.csv")

print("="*50)
print("DATASET SHAPE")
print(df.shape)

print("\nFIRST 5 ROWS")
print(df.head())

print("\nDATASET INFO")
print(df.info())

print("\nSTATISTICAL SUMMARY")
print(df.describe())

# =========================
# DATA CLEANING
# =========================

if 'Id' in df.columns:
    df.drop('Id', axis=1, inplace=True)

print("\nMISSING VALUES")
print(df.isnull().sum())

# =========================
# TARGET ANALYSIS
# =========================

print("\nQUALITY DISTRIBUTION")
print(df['quality'].value_counts())

plt.figure(figsize=(8,5))
sns.countplot(x='quality', data=df)
plt.title("Wine Quality Distribution")
plt.show()

# =========================
# CORRELATION HEATMAP
# =========================

plt.figure(figsize=(12,8))
sns.heatmap(
    df.corr(),
    annot=True,
    cmap='coolwarm'
)
plt.title("Feature Correlation Heatmap")
plt.show()

# =========================
# QUALITY CLASSIFICATION
# =========================

# Good Wine = 1
# Bad Wine = 0

df['quality'] = df['quality'].apply(
    lambda x: 1 if x >= 7 else 0
)

print("\nGOOD/BAD WINE COUNTS")
print(df['quality'].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(x='quality', data=df)
plt.title("Good vs Bad Wine")
plt.show()

# =========================
# FEATURE SELECTION
# =========================

X = df.drop('quality', axis=1)
y = df['quality']

# =========================
# FEATURE SCALING
# =========================

scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================
# RANDOM FOREST
# =========================

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("\n" + "="*50)
print("RANDOM FOREST RESULTS")
print("="*50)
print("Accuracy:", rf_acc)

print(classification_report(
    y_test,
    rf_pred
))

# =========================
# SGD CLASSIFIER
# =========================

sgd = SGDClassifier(
    random_state=42
)

sgd.fit(X_train, y_train)

sgd_pred = sgd.predict(X_test)

sgd_acc = accuracy_score(y_test, sgd_pred)

print("\n" + "="*50)
print("SGD CLASSIFIER RESULTS")
print("="*50)
print("Accuracy:", sgd_acc)

print(classification_report(
    y_test,
    sgd_pred
))

# =========================
# SUPPORT VECTOR CLASSIFIER
# =========================

svc = SVC(
    kernel='rbf',
    random_state=42
)

svc.fit(X_train, y_train)

svc_pred = svc.predict(X_test)

svc_acc = accuracy_score(y_test, svc_pred)

print("\n" + "="*50)
print("SUPPORT VECTOR CLASSIFIER RESULTS")
print("="*50)
print("Accuracy:", svc_acc)

print(classification_report(
    y_test,
    svc_pred
))

# =========================
# MODEL COMPARISON
# =========================

results = pd.DataFrame({
    'Model': [
        'Random Forest',
        'SGD',
        'SVC'
    ],
    'Accuracy': [
        rf_acc,
        sgd_acc,
        svc_acc
    ]
})

print("\nMODEL COMPARISON")
print(results)

plt.figure(figsize=(8,5))
sns.barplot(
    data=results,
    x='Model',
    y='Accuracy'
)
plt.title("Model Accuracy Comparison")
plt.ylim(0,1)
plt.show()

# =========================
# BEST MODEL
# =========================

best_model_name = results.loc[
    results['Accuracy'].idxmax(),
    'Model'
]

print("\nBEST MODEL:", best_model_name)

# =========================
# CONFUSION MATRIX
# =========================

best_pred = rf_pred

if best_model_name == "SGD":
    best_pred = sgd_pred

elif best_model_name == "SVC":
    best_pred = svc_pred

cm = confusion_matrix(
    y_test,
    best_pred
)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)
plt.title(
    f"Confusion Matrix ({best_model_name})"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# =========================
# FEATURE IMPORTANCE
# =========================

rf.fit(X_train, y_train)

importance = pd.DataFrame({
    'Feature': X.columns if hasattr(X, 'columns') else df.drop('quality',axis=1).columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTOP FEATURES")
print(importance.head(10))

plt.figure(figsize=(10,6))
sns.barplot(
    data=importance.head(10),
    x='Importance',
    y='Feature'
)
plt.title("Top 10 Important Features")
plt.show()

# =========================
# SAMPLE PREDICTION
# =========================

sample = X_test[0].reshape(1,-1)

prediction = rf.predict(sample)[0]

if prediction == 1:
    print("\nPrediction: GOOD QUALITY WINE")
else:
    print("\nPrediction: BAD QUALITY WINE")

FileNotFoundError: [Errno 2] No such file or directory: 'WineQT.csv'